# Importing XDF files in HyPyP
@Author : [@jonasmago](https://github.com/jonasmago) & [@FranckPrts](https://github.com/FranckPrts).

Updated by : [@patricefortin](https://github.com/patricefortin) 

Last updated: 2025/6/6

This tutorial guides you in using the `XDFImport` class provided by [HyPyP](https://github.com/ppsp-team/HyPyP/blob/master/hypyp/xdf/).

[XDF files](https://github.com/sccn/xdf) are generally produced by [LSL](https://labstreaminglayer.readthedocs.io/index.html) (Lab Streaming Layer) when recording multi-stream time-series data generated in different modalities (e.g., EEG, video, audio). This open-source format enables associating extensive meta information. It is tailored for biosignal data (e.g., EEG, EOG, ECG, MEG) but still supports high sampling rate data (e.g., audio) or high-channel data (e.g., video, fMRI).

In this tutorial, we convert 2 EEG streams from a sample XDF into 2 `mne.io.RawArray`.


## Imports & path to XDF file

In [ ]:
%matplotlib inline


In [ ]:
from hypyp.xdf import XDFImport
from hypyp import datasets

# Two Starstim-32 headsets recording pure noise (no participants)
path_xdf = datasets.xdf_dyad_noise()

# Synthetic noise with event markers
path_xdf_with_markers = datasets.xdf_dyad_with_markers()

First, we instanciate the class without doing any conversion, to inspect the content of the XDF streams: 

In [ ]:
xdf = XDFImport(path_xdf, convert_to_mne=False)
print(xdf)

**Great!** Now we know what our XDF is composed of. Let's explore different ways of converting on or multiple EEG stream from it into `mne.io.RawArray` object(s).

### Two ways to use `XDFImport`

Here we look into two examples where we use `IMPORT_XDF` to convert the EEG streams into `mne.Raw`(s):
- **Situation 1:** The user wants to (blindly) **convert all** available EEG stream(s) in the XDF into `mne.Raw`(s)
- **Situation 2:** The user knows the index and/or name of one/multiple EEG stream(s) to convert

#### Situation 1
The user wants to **convert all** available EEG stream(s) in the XDF into `mne.io.RawArray`(s). This will convert all the signal streams that are compatible with MNE, and also add markers as annotations to every `mne.io.RawArray` objects

In [ ]:
xdf = XDFImport(path_xdf)
xdf.mne_raws_dict

In [ ]:
# A file with markers
xdf = XDFImport(path_xdf_with_markers)
print(xdf)
print(xdf.mne_raws_dict)
print(xdf.mne_raws[0].annotations)

#### Situation 2
We know either the identifier (integers) of the EEG streams we want to convert, or we know their name. Here using both the chanel indexes and names.

In [ ]:
xdf = XDFImport(path_xdf, select_matches=["LSLOutletHS1-EEG", "LSLOutletHS2-EEG"])
xdf.mne_raws_dict

In [ ]:
xdf = XDFImport(path_xdf, select_matches=[2, 7], scale=1e-6)
xdf.mne_raws_dict

# Extracting the XDFImport's output(s)

A practical way to save the converted stream to mne.Raw is to set a path for the class `XDFImport` to save the .fif file of each stream


In [ ]:
# dictionary of mne raws
xdf.mne_raws_dict

In [ ]:
# list of mne raws
xdf.mne_raws


To extract one of the `mne.Raw` object form the `eeg` class, you can do the following:

In [ ]:
subject1_raw = xdf.mne_raws_dict["LSLOutletHS1-EEG"]
subject1_raw

The name of the original XDF stream name is also stored in the `subject_info` dictionary nested in `Raw.Info`. So even if you separate a `Raw` from its key, you can still find the stream name's in its info.

In [ ]:
# Print the name of the stream from the Raw.info object itself
print(f"subject1 info: {subject1_raw.info['subject_info']['his_id']}")

Once extracted, you can perform any analysis to the `standalone_raw` as you would with a `mne.Raw`!

In [ ]:
subject1_raw.plot(scalings='auto')

## Save the imported data to .fif files


In [ ]:
from pathlib import Path

results_dir = Path('results')
results_dir.mkdir(parents=True, exist_ok=True)

saved_files = xdf.save_to_fif_files(str(results_dir))
print(f'Files: {saved_files}')

## Montage

`XDFImport` does not set the montage automatically! Now that we have converted our stream(s) to `mne.Raw` object(s), we can fix that by setting the montage following [mne's general instruction for setting montage](https://mne.tools/stable/auto_tutorials/intro/40_sensor_locations.html#about-montages-and-layouts). For example:

In [ ]:
# Set the montage to the standard 10-20 montage
xdf.rename_channels(['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T7', 'T8', 'P7', 'P8', 'Fz', 'Cz', 'Pz', 'POz', 'FC1', 'FC2', 'CP1', 'CP2', 'FC5', 'FC6', 'CP5', 'CP6', 'FT9', 'FT10', 'TP9', 'TP10'])
xdf.set_montage('standard_1020')

# Plot the sensors
xdf.mne_raws_dict['LSLOutletHS1-EEG'].plot_sensors(show_names=True)

### Using markers


In [ ]:
xdf = XDFImport(path_xdf_with_markers, scale=1e-6)

for raw in xdf.mne_raws:
    _ = raw.plot(scalings='auto')


### Using sample XDF data from MNE


In [ ]:
# See https://mne.tools/stable/auto_examples/io/read_xdf.html
from mne.datasets import misc

file_path = misc.data_path() / "xdf" / "sub-P001_ses-S004_task-Default_run-001_eeg_a2.xdf"

xdf_misc = XDFImport(file_path, scale=10e-9)

print(xdf_misc)

xdf_misc.mne_raws[0].plot(duration=1, start=14)